# UVA + CoT/LLM wrapper → LIBERO-10 평가 (Colab)

동결된 UVA(`libero10.ckpt`) 위에 **추론 시점 CoT 오케스트레이션**을 씌워 LIBERO-10을 평가합니다.

| 모드 | 설명 |
|------|------|
| `no_cot=True` | 논문 baseline (`eval_sim.py`와 동일) |
| `planner="rule"` | 규칙 기반 CoT → `language_goal` 재작성 (API 불필요) |
| `planner="llm"` | OpenAI vision CoT (`gpt-4o-mini` 등, **API 키 필요**) |

**비교 목표**: baseline vs rule-CoT vs LLM-CoT 의 `test_mean_score`.

> 이미 `colab_libero10_eval.ipynb`로 4b까지 설치했다면 **셀 0 → 8(OpenAI 키) → 9(함수) → 10~** 만 실행해도 됩니다.

> **런타임**: GPU (L4/T4/A100). `MUJOCO_GL=egl` 필수. LLM 모드는 Colab Secrets에 `OPENAI_API_KEY` 등록.

## 0. 런타임 체크 + headless GL

In [ ]:
import os, subprocess
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MKL_THREADING_LAYER"] = "GNU"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 1. (선택) Google Drive
`USE_DRIVE=True`면 체크포인트·데이터·결과를 Drive에 캐시합니다.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT = "/content/drive/MyDrive/uva_libero"
    os.makedirs(PERSIST_ROOT, exist_ok=True)
    print("Drive:", PERSIST_ROOT)
else:
    PERSIST_ROOT = None
    print("Drive 미사용")

## 2. apt (EGL / MuJoCo)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libegl1 libegl1-mesa libgles2-mesa libgl1-mesa-glx \
    libosmesa6 libosmesa6-dev libglfw3 libglew-dev patchelf ffmpeg > /dev/null

## 3. repo 클론
**CoT/LLM 코드가 있는 fork**를 클론하세요 (`unified_video_action/cot/`, `eval_sim_cot.py` 포함).

In [ ]:
import os

UVA_REPO_URL    = "https://github.com/zser99/Unified-Video-Action-with-LLM.git"
UVA_REPO_BRANCH = "main"

%cd /content
if not os.path.isdir("/content/unified_video_action"):
    !git clone --depth 1 -b {UVA_REPO_BRANCH} {UVA_REPO_URL} unified_video_action
if not os.path.isdir("/content/LIBERO"):
    !git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
!ls /content/unified_video_action/unified_video_action/cot 2>/dev/null || echo "WARNING: cot/ folder missing — fork URL 확인"

## 4a. pip 다운그레이드 (gym 0.21)
실행 후 **Runtime → Restart session** → 셀 0 재실행 → **4b** 실행.

In [ ]:
import sys, subprocess
print("Python:", sys.version)
!apt-get install -y -qq python3-dev build-essential pkg-config \
    libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev > /dev/null
!pip install -q "pip==21.3.1" "setuptools==65.5.0" "wheel==0.38.4"
subprocess.run([sys.executable, "-m", "pip", "--version"], check=False)
print("4a 완료 → 재시작 후 4b")

### 재시작 후 4b부터 계속

## 4b. 패키지 설치 (+ openai for LLM CoT)

In [ ]:
import sys, subprocess, tempfile, os, tarfile

def _pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

_pip("install", "-q", "pip==21.3.1", "setuptools==65.5.0", "wheel==0.38.4")

def install_gym_021():
    r = _pip("install", "-q", "gym==0.21.0", check=False)
    if r.returncode == 0:
        import gym; print("gym OK:", gym.__version__); return
    td = tempfile.mkdtemp()
    _pip("download", "--no-deps", "-d", td, "gym==0.21.0")
    tgz = [f for f in os.listdir(td) if f.endswith(".tar.gz")][0]
    root = os.path.join(td, "extract"); os.makedirs(root, exist_ok=True)
    tarfile.open(os.path.join(td, tgz)).extractall(root)
    src = os.path.join(root, [d for d in os.listdir(root) if d.startswith("gym")][0])
    setup_py = os.path.join(src, "setup.py")
    text = open(setup_py, encoding="utf-8").read().replace("opencv-python>=3.", "opencv-python>=3.0")
    open(setup_py, "w", encoding="utf-8").write(text)
    _pip("install", "-q", src)
    import gym; print("gym OK (patched):", gym.__version__)

install_gym_021()

_pip("install", "-q",
     "numpy==1.26.4", "scipy==1.11.4", "numba==0.60.0",
     "hydra-core==1.2.0", "omegaconf==2.2.3", "dill==0.3.7", "einops==0.6.1",
     "diffusers==0.18.2", "transformers==4.28.0", "huggingface_hub==0.25.2",
     "timm==0.9.7", "accelerate==1.0.1", "zarr==2.16.1", "numcodecs==0.11.0",
     "imagecodecs==2024.12.30", "kornia==0.8.0", "opencv-python==4.11.0.86",
     "h5py", "wandb==0.18.3", "gdown==5.2.0", "click", "pandas",
     "pillow", "openai>=1.40.0")
_pip("install", "-q", "av", check=False)
_pip("install", "-q", "mujoco==3.1.6", "robosuite==1.4.1")
_pip("install", "-q", "robomimic==0.3.0", check=False)
_pip("install", "-q", "-e", "/content/LIBERO")
_pip("install", "-q", "-e", "/content/unified_video_action", "--no-deps")
print("4b 완료")

## 5. 체크포인트 (~3 GB)

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")
%cd /content/unified_video_action
os.makedirs("checkpoints", exist_ok=True)
CKPT = "checkpoints/libero10.ckpt"
if not os.path.isfile(CKPT):
    !gdown 11c2VrmaRp48yw__5A5xpcu8EPzkexHSi -O {CKPT}
!ls -lh checkpoints

## 6. LIBERO-10 hdf5

In [ ]:
import glob
%cd /content/unified_video_action
os.makedirs("data/libero_10", exist_ok=True)
if len(glob.glob("data/libero_10/*.hdf5")) < 10:
    !gdown 1_6Kc7e-s30MblbX8YjpxSofe9ZRPk3xv -O data/libero_10_raw.zip
    !unzip -oq data/libero_10_raw.zip -d data/
    for cand in ["libero10", "Libero10"]:
        p = f"data/{cand}"
        if os.path.isdir(p):
            !mv {p}/*.hdf5 data/libero_10/ 2>/dev/null || true
    !rm -f data/libero_10_raw.zip
hdf5 = sorted(glob.glob("data/libero_10/*.hdf5"))
print(len(hdf5), "files"); assert len(hdf5) == 10

## 7. import 검증 (+ CoT 모듈)

In [ ]:
import sys, os, importlib.util
os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, "/content/unified_video_action")
_missing = [m for m in ("hydra", "gym", "robomimic", "mujoco", "libero", "openai")
            if importlib.util.find_spec(m) is None]
assert not _missing, f"설치 누락: {_missing} — 4a/4b 다시 실행"

from unified_video_action.cot.factory import create_planner
from unified_video_action.policy.cot_orchestrated_policy import CoTOrchestratedPolicy
print("CoT modules OK")

## 8. OpenAI API 키 (LLM CoT용)
Colab: 왼쪽 **열쇠 아이콘 → Secrets** → `OPENAI_API_KEY` 추가.

규칙 CoT(`planner="rule"`)만 쓸 때는 이 셀을 건너뛰어도 됩니다.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY: loaded from Colab Secrets (", len(os.environ["OPENAI_API_KEY"]), "chars)")
except Exception as e:
    print("Secrets 없음:", e)
    print("수동 설정: os.environ['OPENAI_API_KEY'] = 'sk-...'")
# os.environ["OPENAI_API_KEY"] = "sk-..."  # 필요 시 주석 해제

## 9. `run_libero10_cot_eval` — `eval_sim_cot.py` 인라인

핵심 인자:
- `no_cot=True` → baseline
- `planner="rule"|"llm"`
- `replan_every=8` (기본), `verbose_cot=True`면 replan마다 trace 출력

**셀 9a (선택)**: MuJoCo 없이 CoT planner만 API/규칙 테스트 — smoke 전에 30초 점검용.

In [ ]:
# 9a — CoT planner만 빠르게 검증 (시뮬레이터 / 체크포인트 불필요)
import os
from unified_video_action.cot.factory import create_planner

_test_goal = "put both moka pots on the stove"
for _name in ("rule", "llm"):
    p = create_planner(_name, model="gpt-4o-mini", fallback_rule_based=True)
    plan = p.plan(base_goal=_test_goal, step_index=0, replan_index=0, obs_dict=None, num_candidates=1)
    print(f"\n=== planner={_name} ===")
    print("language_goal:", plan.language_goal[:120], "...")
    print(plan.cot_trace[:300])

In [ ]:
import os, sys, json, glob, random, pathlib, dill, hydra, torch, numpy as np, wandb
from omegaconf import open_dict

%cd /content/unified_video_action
sys.path.insert(0, "/content/unified_video_action")

from unified_video_action.workspace.base_workspace import BaseWorkspace
from unified_video_action.env_runner.base_image_runner import BaseImageRunner
from unified_video_action.cot.factory import create_planner
from unified_video_action.policy.cot_orchestrated_policy import CoTOrchestratedPolicy


def _instantiate_libero_runners(cfg, output_dir, task_subset):
    hdf5_files = sorted(glob.glob(cfg.task.dataset.dataset_path + "/*hdf5"))
    if task_subset:
        hdf5_files = [f for f in hdf5_files if any(s in os.path.basename(f) for s in task_subset)]
        assert hdf5_files, f"task_subset {task_subset} matched nothing"
    print(f"Tasks ({len(hdf5_files)}):")
    for f in hdf5_files:
        print(" ", os.path.basename(f))
    runners = []
    for f in hdf5_files:
        r = hydra.utils.instantiate(cfg.task.env_runner, task_dir=f, output_dir=output_dir)
        assert isinstance(r, BaseImageRunner)
        runners.append(r)
    return runners


def run_libero10_cot_eval(
    checkpoint="checkpoints/libero10.ckpt",
    output_dir="outputs/libero10_cot",
    device="cuda:0",
    n_test=1,
    n_train=0,
    n_envs=1,
    n_test_vis=0,
    max_steps=None,
    task_subset=None,
  # --- CoT ---
    no_cot=False,
    planner="llm",
    llm_model="gpt-4o-mini",
    replan_every=8,
    num_candidates=1,
    candidate_strategy="first",
    verbose_cot=False,
    no_llm_fallback=False,
):
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)
    if device.startswith("cuda") and not torch.cuda.is_available():
        print("WARNING: no CUDA, using cpu"); device = "cpu"

    payload = torch.load(open(checkpoint, "rb"), pickle_module=dill, map_location="cpu")
    cfg = payload["cfg"]
    seed = cfg.training.seed
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    with open_dict(cfg):
        cfg.output_dir = output_dir
        cfg.task.env_runner.n_test = int(n_test)
        cfg.task.env_runner.n_train = int(n_train)
        cfg.task.env_runner.n_envs = int(n_envs)
        cfg.task.env_runner.n_test_vis = min(n_test_vis, n_test)
        cfg.task.env_runner.n_train_vis = 0
        if max_steps is not None:
            cfg.task.env_runner.max_steps = int(max_steps)

    workspace = hydra.utils.get_class(cfg.model._target_)(cfg, output_dir=output_dir)
    workspace.load_payload(payload, exclude_keys=None, include_keys=None)
    inner = workspace.ema_model
    inner.to(device); inner.eval()

    policy = inner
    cot_meta = {"cot_enabled": False}
    if not no_cot:
        cot_planner = create_planner(
            planner, model=llm_model, fallback_rule_based=not no_llm_fallback
        )
        policy = CoTOrchestratedPolicy(
            inner_policy=inner,
            planner=cot_planner,
            replan_every=replan_every,
            num_candidates=num_candidates,
            candidate_strategy=candidate_strategy,
            verbose=verbose_cot,
        )
        cot_meta = {
            "cot_enabled": True,
            "planner": planner,
            "replan_every": replan_every,
            "num_candidates": num_candidates,
            "candidate_strategy": candidate_strategy,
        }
        if planner == "llm":
            cot_meta["llm_model"] = llm_model

    env_runners = _instantiate_libero_runners(cfg, output_dir, task_subset)
    step_log = {}
    for er in env_runners:
        step_log.update(er.run(policy))
        try: er.env.close()
        except Exception: pass

    test_scores = {k: v for k, v in step_log.items() if "test/" in k and "_mean_score" in k}
    step_log["test_mean_score"] = float(np.mean(list(test_scores.values()))) if test_scores else float("nan")

    json_log = {}
    for k, v in step_log.items():
        if isinstance(v, wandb.sdk.data_types.video.Video):
            json_log[k] = v._path
        else:
            try: json.dumps(v); json_log[k] = v
            except TypeError: json_log[k] = str(v)
    json_log.update(cot_meta)
    json_log["device"] = device
    tag = "baseline" if no_cot else planner
    out_path = os.path.join(output_dir, f"eval_cot_{tag}_{os.path.basename(checkpoint)}.json")
    with open(out_path, "w") as f:
        json.dump(json_log, f, indent=2, sort_keys=True)
    print("Saved", out_path)
    print("test_mean_score:", step_log["test_mean_score"])
    return step_log, json_log

## 10. Smoke — 1 task × 1 ep (baseline / rule / LLM 비교)
약 30–60분 (LLM은 API 호출 + 시뮬). `task_subset`은 본인 환경에 맞게 바꾸세요.

In [ ]:
SMOKE_TASK = ["moka_pot"]
SMOKE_KW = dict(n_test=1, n_train=0, n_envs=1, n_test_vis=0, max_steps=300, task_subset=SMOKE_TASK)

step_baseline, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/baseline",
    no_cot=True,
    **SMOKE_KW,
)

step_rule, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/rule",
    planner="rule",
    verbose_cot=True,
    **SMOKE_KW,
)

step_llm, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/llm",
    planner="llm",
    llm_model="gpt-4o-mini",
    verbose_cot=True,
    **SMOKE_KW,
)

## 11. Light — 10 tasks × 3 ep (LLM CoT, 논문 30-test 규모)
L4 기준 약 2–4시간 + OpenAI API 비용. baseline 비교는 **셀 25(smoke)** 에서 이미 `no_cot=True` 가 포함됩니다.

In [ ]:
step_light_llm, json_light_llm = run_libero10_cot_eval(
    output_dir="outputs/cot_light/llm",
    planner="llm",
    llm_model="gpt-4o-mini",
    n_test=3,
    n_envs=1,
    n_test_vis=0,
    verbose_cot=False,
)
print("LLM light mean:", step_light_llm.get("test_mean_score"))

## 12. 결과 비교표

In [ ]:
import pandas as pd

rows = []
for name, log in [
    ("baseline", locals().get("step_baseline")),
    ("rule-CoT", locals().get("step_rule")),
    ("LLM-CoT smoke", locals().get("step_llm")),
    ("LLM-CoT light", locals().get("step_light_llm")),
]:
    if log is None:
        continue
    rows.append({"mode": name, "test_mean_score": log.get("test_mean_score", float("nan"))})

if rows:
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\n논문 Table I (UVA baseline, 50 ep): 0.90")
    print("논문 Suppl. VIII (30 ep): 0.93")
else:
    print("아직 평가 결과 없음 — 셀 10 이상 실행")

## 트러블슈팅
- **`cot/` 없음** → 셀 3 fork URL이 LLM wrapper 브랜치인지 확인.
- **LLM이 rule로 fallback** → `OPENAI_API_KEY` 미설정. 셀 8 + trace에 `[LLM skipped]` 확인.
- **gym 설치 실패** → `colab_libero10_eval.ipynb`와 동일: 4a → 재시작 → 4b.
- **API 비용** → smoke는 1 task·1 ep; light는 replan마다 1회 API 호출 × (steps/8) × 30 ep.